In [1]:
import numpy as np
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)

print("downloading fashion MNIST dataset(this may take 30-60 seconds)...")

fashion_mnist=fetch_openml('Fashion-MNIST',version=1,as_frame=False)

x=fashion_mnist.data
y=fashion_mnist.target.astype(int)

print(f"total dataset size:{x.shape[0]} images,each with {x.shape[1]} pixels.")

x_subset,_,y_subset,_=train_test_split(x,y,train_size=12000,stratify=y,random_state=42)
x_train,x_test,y_train,y_test=train_test_split(x_subset,y_subset,test_size=2000,stratify=y_subset,random_state=42)
print(f"Training image : {x_train.shape[0]}")
print(f"Testing image: {x_test.shape[0]}")

print(f"before scaling -> Min: {x_train.min()},Max: {x_train.max()}")
x_train=x_train/255.0
x_test=x_test/255.0
print(f"after scaling  -> Min: {x_train.min()},Max: {x_train.max()}")

k_values={1,3,5,7,9,15}
results={}
print(f"{'k Values':<8} |{'Accuarcy':<10} | {'prediction time (seconds)':<25}")
print("_"*50)

for k in k_values:
    knn=KNeighborsClassifier(n_neighbors=k,metric='euclidean',n_jobs=-1)
    knn.fit(x_train,y_train)
    start_time=time.time()
    y_pred=knn.predict(x_test)
    elapsed_time=time.time()-start_time
    acc=accuracy_score(y_test,y_pred)
    results[k]={
    "accuracy":acc,
    "time":elapsed_time,
    "predictions":y_pred
    }
    print(f"{k:<8} | {acc* 100:<9.2f}% | {elapsed_time:<25.2f}")


class_names={
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "sandal","shirt","sneaker","bag","ankle boot"
}
best_k=max(results,key=lambda k: results[k]["accuracy"])
print(f"Best K is : {best_k} with {results[best_k]['accuracy']*100:.2f} %accuracy\n")

print("pers-Class Classification Report :")
print(classification_report(y_test,results[best_k]["predictions"],target_names=class_names))

downloading fashion MNIST dataset(this may take 30-60 seconds)...
total dataset size:70000 images,each with 784 pixels.
Training image : 10000
Testing image: 2000
before scaling -> Min: 0,Max: 255
after scaling  -> Min: 0.0,Max: 1.0
k Values |Accuarcy   | prediction time (seconds)
__________________________________________________
1        | 79.30    % | 1.72                     
3        | 80.90    % | 0.20                     
5        | 81.00    % | 0.20                     
7        | 81.10    % | 0.20                     
9        | 80.95    % | 0.20                     
15       | 80.25    % | 0.22                     
Best K is : 7 with 81.10 %accuracy

pers-Class Classification Report :
              precision    recall  f1-score   support

 T-shirt/top       0.73      0.83      0.78       200
     sneaker       0.97      0.94      0.96       200
        Coat       0.67      0.73      0.70       200
       shirt       0.85      0.83      0.84       200
    Pullover       0.74  